# 3 — Embeddings

LLMs can generate text, but they can also turn text into **vectors** — lists of numbers that capture meaning.

These vectors live in a high-dimensional space where **similar meanings are close together**.

We'll cover:
1. Embedding a word and looking at the raw vector
2. Comparing two embeddings with cosine similarity
3. Trying different word pairs to build intuition

In [ ]:
from dotenv import load_dotenv
from mistralai import Mistral
import numpy as np
import os

load_dotenv()

client = Mistral(api_key=os.getenv("mistral_api_key"))

## Embed a single word

Let's turn the word "dog" into a vector and see what we get.

In [ ]:
response = client.embeddings.create(
    model="mistral-embed",
    inputs=["dog"],
)

vector = response.data[0].embedding

print(f"Type: {type(vector)}")
print(f"Dimensions: {len(vector)}")
print(f"First 10 values: {vector[:10]}")

That's a list of 1024 floating-point numbers. One word → 1024 numbers. A whole paragraph gets compressed into the same 1024 numbers.

On their own these numbers are meaningless. The magic happens when you **compare** two vectors.

## Cosine similarity

Cosine similarity measures the angle between two vectors:
- **1.0** — identical meaning
- **0.0** — unrelated
- **-1.0** — opposite meaning (rare in practice)

Let's compare "dog" and "cat":

In [ ]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
response = client.embeddings.create(
    model="mistral-embed",
    inputs=["dog", "cat"],
)

vec_dog = response.data[0].embedding
vec_cat = response.data[1].embedding

sim = cosine_similarity(vec_dog, vec_cat)
print(f"dog vs cat: {sim:.4f}")

High similarity — makes sense, they're both pets/animals.

Now let's try something less related:

In [ ]:
response = client.embeddings.create(
    model="mistral-embed",
    inputs=["dog", "bicycle"],
)

sim = cosine_similarity(response.data[0].embedding, response.data[1].embedding)
print(f"dog vs bicycle: {sim:.4f}")

Lower similarity. The model understands that a dog and a bicycle have less in common than a dog and a cat.

## It works on sentences too

Embeddings aren't just for single words — they capture meaning at any length.

In [ ]:
sentences = [
    "The cat sat on the mat",
    "A kitten was resting on the rug",
    "Stock prices rose sharply on Monday",
]

response = client.embeddings.create(
    model="mistral-embed",
    inputs=sentences,
)

vecs = [item.embedding for item in response.data]

print(f"cat/mat vs kitten/rug:  {cosine_similarity(vecs[0], vecs[1]):.4f}")
print(f"cat/mat vs stock prices: {cosine_similarity(vecs[0], vecs[2]):.4f}")

The two cat sentences are close together even though they share almost no words. The stock market sentence is far away.

This is **semantic** similarity, not keyword matching. That's the key idea.

## Try your own

Change the words/sentences below and see what happens:

In [ ]:
text_a = ""
text_b = ""

response = client.embeddings.create(
    model="mistral-embed",
    inputs=[text_a, text_b],
)

sim = cosine_similarity(response.data[0].embedding, response.data[1].embedding)
print(f"\"{text_a}\" vs \"{text_b}\": {sim:.4f}")